# Calling qodec instructions from OpenQASM

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do OpenQASM 3 gates and measurements resolve to top-layer qodec instructions?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Declare custom `qodec` block operations in OpenQASM 3 using `@qdk.qir.intrinsic`.
- Map OpenQASM `measure` statements to `__quantum__qis__m__body` and understand runtime flag handling.
- Author and audit a compact multi-document YAML `qodec` bundle for OpenQASM programs.

## Setting up

The cell below imports `qdk.openqasm`, `run_qir`, and a fresh copy of the upgraded `C4` protocol.

In [ ]:
import qodec
from qdk import TargetProfile, ec, openqasm
from qdk.simulation import run_qir
from _unit import make_upgraded_protocol

protocol = make_upgraded_protocol()

## Mapping OpenQASM 3 statements to qodec mnemonics

Just as in Q#, OpenQASM 3 compiles to QIR that resolves against the exact same `qodec` top layer, with four current lowering rules to keep in mind (like Q# intrinsics, these low-level QIR naming conventions reflect today's front-end binding surface and may be streamlined in future QDK releases):

| OpenQASM 3 syntax | Top-layer `qodec` mnemonic | Behavior |
| :--- | :--- | :--- |
| `@qdk.qir.intrinsic` followed by `gate foo q {}` | `foo` | Calls custom block preparations or unitaries (`@qdk.qir.intrinsic` must sit on the line above `gate` or `def`). |
| `@qdk.qir.intrinsic` followed by `def turn(float theta, qubit q) {}` | `turn` | Binds `float`, `int`, and `bool` arguments to instruction `parameters` (forwarded via `parameter_bindings`). |
| `c = measure q;` | `__quantum__qis__m__body` | Required for readout because OpenQASM custom intrinsics cannot return a QIR `Result`; observes one logical bit. |
| Standard gate (`x q;`, `cx a, b;`, `sdg q;`) | `__quantum__qis__x__body`, `__quantum__qis__cx__body`, `__quantum__qis__s__adj` | Resolves only if the top layer explicitly defines that QIR intrinsic name. |

As in Q#, starting a qubit with an `out`-only instruction (`prepare_z_all block;`) creates its block directly; otherwise the top layer must provide a default `stabilize: Z_0` preparation. Because OpenQASM `gate` declarations cannot return flag bits, any `flags` on an instruction (such as `prepare_z_all`'s `reject` flag) are handled **automatically by the runtime** according to `on_shot_failure`.

Let's add a single-bit `__quantum__qis__m__body` gadget (observing `Z_0`) to a copy of our `C4` protocol and run an OpenQASM 3 program through it:

In [ ]:
qasm_protocol = qodec.Qodec.loads(protocol.dumps())
layer0 = qasm_protocol.layers[0]

m_instr = qodec.Instruction(
    mnemonic="__quantum__qis__m__body",
    inputs=[qodec.instructions.BlockOperand("C4")],
    outputs=[],
    action=[qodec.actions.Observe(["Z_0"])],
)
layer0.instruction_set.instructions["__quantum__qis__m__body"] = m_instr
layer0.gadgets["__quantum__qis__m__body"] = ec.filled(
    qodec.Gadget(
        implements=m_instr,
        inputs=list(layer0.gadgets["measure_z_all"].inputs),
        outputs=[],
        circuit=qodec.gadgets.Circuit(
            format="stim",
            source="M 0 1 2 3\n",
            instruction_set=qasm_protocol.layers[1].instruction_set,
        ),
    )
)

source = """
OPENQASM 3.0;
include "stdgates.inc";
@qdk.qir.intrinsic
gate prepare_z_all q {}
@qdk.qir.intrinsic
gate syndrome q {}
qubit block;
bit c;
prepare_z_all block;
syndrome block;
c = measure block;
"""
qir = str(openqasm.compile(source, target_profile=TargetProfile.Adaptive_RI))
qasm_shots = run_qir(qir, qodec=qasm_protocol, type="stabilizer", shots=6, seed=11)
print("OpenQASM 3 shots:", qasm_shots)

## Authoring a compact YAML qodec bundle for OpenQASM

You can also author an entire multi-document `qodec` directly in YAML with `qodec.Qodec.loads(...)`. Below, `prepare_checked` pairs $\vert 0\rangle$ preparation with an ancilla CNOT check (`flags: [reject]`), `flip` applies Pauli $X$, and `c = measure q;` reads out the result:

In [ ]:
yaml_qodec = qodec.Qodec.loads("""
---
qodec.yaml:
  name: openqasm_checked_demo
  layers:
  - instruction_set: top.isa.yaml
    codes: {qubit: qubit.code.yaml}
    gadgets:
      prepare_checked: prepare_checked.gadget.yaml
      flip: flip.gadget.yaml
      __quantum__qis__m__body: m.gadget.yaml
  - instruction_set: physical.isa.yaml
---
top.isa.yaml:
  name: top
  blocks: {qubit: 1}
  instructions:
  - mnemonic: prepare_checked
    description: Checked |0> preparation.
    out: [qubit]
    action: [stabilize: Z_0]
    flags: [reject]
  - mnemonic: flip
    description: Pauli X.
    in: [qubit]
    out: [qubit]
    action: [pauli: X_0]
  - mnemonic: __quantum__qis__m__body
    description: Measure Z.
    in: [qubit]
    action: [observe: Z_0]
---
physical.isa.yaml:
  name: physical
  blocks: {qubit: 1}
  instructions:
  - {mnemonic: R, description: Reset., out: [qubit], action: [stabilize: Z_0]}
  - {mnemonic: X, description: Pauli X., in: [qubit], out: [qubit], action: [pauli: X_0]}
  - mnemonic: CX
    description: CNOT.
    in: [qubit, qubit]
    out: [qubit, qubit]
    action: [clifford: {X_0: X_0 X_1, Z_1: Z_0 Z_1}]
  - {mnemonic: M, description: Measure Z., in: [qubit], action: [observe: Z_0]}
---
qubit.code.yaml:
  name: qubit
  description: One unencoded qubit.
  stabilizers: []
  x: [X_0]
  z: [Z_0]
---
prepare_checked.gadget.yaml:
  circuit: {source: [R: [0], R: [1], CX: [0, 1], M: [1]]}
  readouts: [reject: ["circuit.readouts[0]"]]
---
flip.gadget.yaml:
  circuit: {source: [X: [0]]}
---
m.gadget.yaml:
  circuit: {source: [M: [0]]}
  readouts: [["circuit.readouts[0]", "in[0].z[0]"]]
""")

print("Audit:", ec.audit(yaml_qodec))
caller = """
OPENQASM 3.0;
@qdk.qir.intrinsic
gate prepare_checked q {}
@qdk.qir.intrinsic
gate flip q {}
qubit q; bit c;
prepare_checked q;
flip q;
c = measure q;
"""
caller_qir = str(openqasm.compile(caller, target_profile=TargetProfile.Adaptive_RI))
caller_shots = run_qir(caller_qir, qodec=yaml_qodec, type="stabilizer", shots=5, seed=9)
print("prepare_checked -> flip -> measure:", caller_shots)

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;When do you need Q# instead of OpenQASM 3 to call a qodec instruction?</summary>

<div style="padding:0.1em 1em;">

Use Q# `@Measurement()` operations whenever an instruction returns multiple measurement outcomes from a single block (such as `measure_z_all` returning two bits) or when the logical program needs to inspect a returned flag directly.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;What happens when an OpenQASM 3 intrinsic gate triggers a raised reject flag at runtime?</summary>

<div style="padding:0.1em 1em;">

Because OpenQASM `gate` calls do not consume flags as return values, the runtime handles the raised flag automatically using `on_shot_failure` (`'discard'` by default).

</div>

</details></div>

## Exercise: Execute a two-block OpenQASM 3 program on C4

Complete `openqasm_two_block_shots` so it compiles the two-block OpenQASM 3 program (`prepare_z_all`, `cx_all`, `syndrome`, and `measure` on two `C4` blocks) and executes `10` noiseless shots on `qasm_proto` using `run_qir(..., type="stabilizer")`.

In [ ]:
from qdk import TargetProfile, openqasm
from qdk.simulation import run_qir
from _unit import exercise, make_openqasm_c4_protocol


@exercise
def openqasm_two_block_shots() -> int:
    qasm_proto = make_openqasm_c4_protocol()
    qasm_src = '''
    OPENQASM 3.0;
    include "stdgates.inc";
    @qdk.qir.intrinsic
    gate prepare_z_all q {}
    @qdk.qir.intrinsic
    gate cx_all a, b {}
    @qdk.qir.intrinsic
    gate syndrome q {}
    qubit b0; qubit b1; bit c0; bit c1;
    prepare_z_all b0; prepare_z_all b1;
    cx_all b0, b1;
    syndrome b0; syndrome b1;
    c0 = measure b0; c1 = measure b1;
    '''
    qir = str(openqasm.compile(qasm_src, target_profile=TargetProfile.Adaptive_RI))
    # Execute 10 noiseless stabilizer shots with run_qir and return len(shots)
    return ...

**Hint**

Call `shots = run_qir(qir, qodec=qasm_proto, type='stabilizer', shots=10, seed=11)` and return `len(shots)`.

In [ ]:
from qdk import TargetProfile, openqasm
from qdk.simulation import run_qir
from _unit import exercise, make_openqasm_c4_protocol


@exercise
def openqasm_two_block_shots() -> int:
    qasm_proto = make_openqasm_c4_protocol()
    qasm_src = '''
    OPENQASM 3.0;
    include "stdgates.inc";
    @qdk.qir.intrinsic
    gate prepare_z_all q {}
    @qdk.qir.intrinsic
    gate cx_all a, b {}
    @qdk.qir.intrinsic
    gate syndrome q {}
    qubit b0; qubit b1; bit c0; bit c1;
    prepare_z_all b0; prepare_z_all b1;
    cx_all b0, b1;
    syndrome b0; syndrome b1;
    c0 = measure b0; c1 = measure b1;
    '''
    qir = str(openqasm.compile(qasm_src, target_profile=TargetProfile.Adaptive_RI))
    shots = run_qir(qir, qodec=qasm_proto, type='stabilizer', shots=10, seed=11)
    return len(shots)

`10`. `@qdk.qir.intrinsic` above `gate cx_all a, b {}` compiles to a two-operand QIR call that `run_qir` lowers through the transversal `cx_all` gadget across the two `C4` blocks.

## Further reading

- [Compile OpenQASM programs for qodec execution ↗](https://learn.microsoft.com/en-us/azure/quantum/run-decode-logical-programs#compile-openqasm-programs)
- [Call qodec instructions from OpenQASM (sample notebook) ↗](https://github.com/microsoft/qdk/blob/main/samples/notebooks/qdk_ec/qodec_instructions_from_openqasm.ipynb)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../07-qsharp-compilation/qsharp_compilation.workbook.ipynb" title="Previous unit: Compiling Logical Experiments from Q#" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Compiling Logical Experiments from Q#</span></span></a><a href="../09-simulation-and-decoding/simulation_and_decoding.workbook.ipynb" title="Next unit: Simulating and Decoding Under Noise" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Simulating and Decoding Under Noise</span></span><span aria-hidden="true">&#8250;</span></a></div>